In [1]:
# 설치
# !pip install -qU pypdf

In [2]:
# OCR
# !pip install -qU rapidocr-onnxruntime

In [3]:
# from langchain_community.document_loaders.parsers import RapidOCRBlobParser

In [4]:
# !pip install pdfminer.six

In [5]:
# !pip install langchain-community pymupdf

In [6]:
# !pip install -qU docx2txt

In [7]:
# !pip install unstructured python-pptx

In [8]:
# import nltk
# nltk.download('punkt_tab')

In [9]:
# Reference
# https://m.blog.naver.com/htk1019/223442628204

In [10]:
import bs4

In [11]:
from langchain_core.documents import Document

In [12]:
from langchain_community.document_loaders.parsers import RapidOCRBlobParser

In [13]:
from langchain_community.document_loaders import PyPDFLoader, PyPDFDirectoryLoader, PDFMinerLoader, UnstructuredPDFLoader
from langchain_community.document_loaders import Docx2txtLoader, UnstructuredPowerPointLoader, TextLoader
# from langchain_community.document_loaders import WebBaseLoader

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter

In [15]:
import re
import json
import warnings
warnings.filterwarnings("ignore")

In [16]:
all_raw_text = []

In [17]:
def transform_clean_string(to_replace): 
    """remove unnecessary characters"""

    if isinstance(to_replace, (str)):
        to_replace = to_replace.strip()
        to_replace = re.sub(r'(?<!\.)(\n|\r\n)', ' ', to_replace)
        to_replace = re.sub(r'\t|\\t', ' ', to_replace)
        to_replace = re.sub(r' +', ' ', to_replace)

    # merge all texts
    all_raw_text.append(to_replace)
    
    return to_replace

In [18]:
# --
# PDF Loader
# --
filepath = './files/asiabrief_3-26.pdf'

loader = PyPDFLoader(filepath, mode="page", images_inner_format="markdown-img", images_parser=RapidOCRBlobParser())
# loader = PyPDFDirectoryLoader("data/")
# loader = PDFMinerLoader(pdf_filepath)
# loader = UnstructuredPDFLoader(pdf_filepath, mode="elements")

# data = [Document(page_content=transform_trim_string(text.page_content), metadata=text.metadata) for i, text in enumerate(loader.load())]
data = [Document(page_content=transform_clean_string(s.page_content), metadata=s.metadata) for s in loader.load()]

In [19]:
# --
# DOC Loader
# --
filepath = './files/Sample.docx'

all_raw_text = []

loader = Docx2txtLoader(filepath)
data = [Document(page_content=transform_clean_string(s.page_content), metadata=s.metadata) for s in loader.load()]

In [20]:
# --
# DOC Loader
# --
filepath = './files/test.log'

all_raw_text = []

loader = TextLoader(filepath, encoding="utf-8")
data = [Document(page_content=transform_clean_string(s.page_content), metadata=s.metadata) for s in loader.load()]

In [21]:
print(f'총 분할된 도큐먼트 수: {len(data)}')

총 분할된 도큐먼트 수: 1


In [22]:
# data

In [23]:
# print(f"전체 텍스트 추출: {''.join(all_raw_text)}")

In [24]:
# 2. Split documents into small, manageable chunks
# text_splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=0)
text_splitter = CharacterTextSplitter(chunk_size=3000, chunk_overlap=500, separator = '\n\n')
docs = text_splitter.split_documents(data)

In [25]:
print(f'TextSplitter 후, 총 분할된 도큐먼트 수: {len(docs)}')

TextSplitter 후, 총 분할된 도큐먼트 수: 1


In [26]:
# docs

In [27]:
# -- Directory

In [28]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader, PyPDFLoader

# 확장자별 로더 매핑
loader_mapping = {
    '.log': TextLoader,
    '.pdf': PyPDFLoader,
}

# 매핑된 로더를 loader_cls에 지정
# glob='./*.*'
# loader = DirectoryLoader('./files', glob="**/*.pdf", loader_cls=PyPDFLoader)
# loader = DirectoryLoader(path='./files', glob=['**/*.txt', '**/*.log'])
# docs = loader.load_and_split()

# 1. 각 확장자별로 DirectoryLoader 생성
txt_loader1 = DirectoryLoader('./files', glob='**/*.txt', loader_cls=TextLoader)
txt_loader2 = DirectoryLoader('./files', glob='**/*.log', loader_cls=TextLoader)
pdf_loader = DirectoryLoader('./files', glob='**/*.pdf', loader_cls=PyPDFLoader)

# 2. 문서 로드
txt_docs1 = txt_loader1.load()
txt_docs2 = txt_loader2.load()
pdf_docs = pdf_loader.load()

# 3. 문서 리스트 병합
docs = txt_docs1 + txt_docs2 + pdf_docs

In [29]:
print(f"총 {len(docs)}개의 문서를 불러왔습니다.")

총 6개의 문서를 불러왔습니다.


In [30]:
for s in docs:
    print(s.metadata.get("source"))

files/test.log
files/asiabrief_3-26.pdf
files/asiabrief_3-26.pdf
files/asiabrief_3-26.pdf
files/asiabrief_3-26.pdf
files/asiabrief_3-26.pdf


In [31]:
docs = [Document(page_content=transform_clean_string(s.page_content), metadata=s.metadata) for s in docs]

In [32]:
# docs

In [33]:
docs[0].page_content

'24/10/11 12:35:22 INFO myLogger: 2024-10-11 12:35:22.075 DB Execute for TableName : Test 24/10/11 12:35:38 ERROR Executor: Exception in task 1.98 in stage 0.0 (TID 107) java.lang.OutOfMemoryError: Java heap space'